In [0]:
%pip install pycryptodome
dbutils.library.restartPython()

In [0]:
# 实现 AES 解密并处理优惠券数据
from Crypto.Cipher import AES              
from Crypto.Util.Padding import pad, unpad
import base64                              
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType  
 
# 定义加密解密工具类
class EncryptHelper:
 
    # 与 Java 代码保持一致的 AES 密钥和初始向量（IV）
    AES_KEY = b"1f424e7ee97b453bb84bac1fc9562211"  # 32字节密钥（AES-256）
    AES_IV = b"3de2fd5f7971fc56"                   # 16字节初始向量（AES CBC 模式要求）
 
    @staticmethod
    def encrypt(value: str) -> str:
        """
        使用 AES CBC 模式加密字符串，并返回 Base64 编码结果。
        Java 默认 PKCS5Padding, 这里使用 PKCS7Padding 与之兼容。
        """
        if not value:  # 如果输入为空，直接返回
            return value
 
        try:
            cipher = AES.new(EncryptHelper.AES_KEY, AES.MODE_CBC, EncryptHelper.AES_IV)
            padded_data = pad(value.encode("utf-8"), AES.block_size, style="pkcs7")
            encrypted_bytes = cipher.encrypt(padded_data)
 
            # 将加密结果进行 Base64 编码，返回字符串
            return base64.b64encode(encrypted_bytes).decode("utf-8")
 
        except Exception as e:
            print(f"Encrypt error: {e}")
            return value
 
    @staticmethod
    def decrypt(value: str) -> str:
        """
        使用 AES CBC 模式解密 Base64 编码的字符串，并去除 PKCS7 填充。
        """
        if not value:  # 如果输入为空，直接返回
            return value
        try:
            # 创建 AES CBC 模式的解密器
            cipher = AES.new(EncryptHelper.AES_KEY, AES.MODE_CBC, EncryptHelper.AES_IV)
            # Base64 解码，得到加密字节
            encrypted_bytes = base64.b64decode(value)
            # 执行解密操作
            decrypted_data = cipher.decrypt(encrypted_bytes)
            # 去除 PKCS7 填充，得到原始明文
            unpadded_data = unpad(decrypted_data, AES.block_size, style="pkcs7")
            # 返回 UTF-8 解码后的字符串
            return unpadded_data.decode("utf-8")
 
        except Exception as e:
            print(f"Decrypt error: {e}")
            return value
 
# 定义解密 VIN 的函数
def code2vin(encrypted_vin):
    """
    将加密的 VIN 字段解密为真实 VIN。
    """
    return EncryptHelper.decrypt(encrypted_vin)
 
# 注册 Spark UDF，用于在 Spark SQL 中调用解密逻辑
code2vin_udf = udf(code2vin, StringType())

In [0]:
def write_coupon_table(spark, run_date, db, table_names):
  # Update coupon table 
  # for backfilling
  if run_date < '2025-12-01':
      df_ = spark.sql("""
                      SELECT *
                      FROM datalake_ecommerce.dw_cust_ecommerce_coupon_history
                      """)
  else:
      df_ = spark.sql(f"""
                      SELECT *
                      FROM datalake_ecommerce.dw_cust_ecommerce_coupon_usage
                      where partition_date = '{run_date}'
                      """.format(run_date = run_date))
  
  # 新增一列 vin，通过 UDF 解密 applicable_vin 字段
  df_p = df_.withColumn("vin", code2vin_udf("applicable_vin"))
  
  df_p.createOrReplaceTempView("df_coupon")
  spark.sql("""
            INSERT OVERWRITE TABLE {db}.{mmc_coupons} PARTITION (partition_date = '{run_date}')
            SELECT vin,
            promotion_code,
            coupon_code,
            to_date(start_time) AS start_date,
            to_date(end_time) AS end_date,
            to_date(redeemed_time) AS redeemed_date,
            status
            FROM df_coupon
            """.format(db = db,
                       mmc_coupons = table_names["mmc_coupons"],
                       run_date = run_date)
            )

In [0]:
def write_vin_meid_mapping_table(spark, run_date, db, table_names):
  # Update vin-meid mapping table
  spark.sql("""
            INSERT OVERWRITE TABLE {db}.{vin_meid_mapping} PARTITION (partition_date = '{run_date}')
            WITH masteruser AS
              (
                  SELECT vin, ciamid, operationtype , row_number() OVER (PARTITION BY vin order by timestamp DESC ) AS RN
                  FROM datalake_cpd.dw_vhcl_cpd_auth_history
                  WHERE userrolecode = 'MASTER_USER'
                  AND partition_date <= '{run_date}'
              ),
            masteruser_status AS (
                      SELECT vin
                      ,ciamid
                      ,CASE WHEN operationtype = 'DELETE' THEN 0
                              ELSE 1 END AS masteruser_status
                  FROM masteruser
                  WHERE RN = 1
              )
            select distinct masteruser_status.vin as vin,
            case when ciamid is not null then ciamid else ''  end as ciamid
            from masteruser_status
            left join datalake_cpd.dw_cust_cpd_customerinfo cust
            on masteruser_status.ciamid=cust.ciam_id and cust.vin=masteruser_status.vin
            where masteruser_status=1 and removed=0
            and cust.partition_date <= '{run_date}'
            """.format(db = db,
                       vin_meid_mapping = table_names["vin_meid_mapping"],
                       run_date = run_date)
            )

In [0]:
def write_mmc_product_list_table(spark, db, table_names):
    # Update candidate MMC product list table 
    spark.sql("""
              INSERT OVERWRITE TABLE {db}.{mmc_product}
              SELECT p.code as product_id,
              vp.code as variant_id,
              p.name as product_name,
              CASE WHEN p.product_type = 1 THEN 'MB'
              WHEN p.product_type = 2 AND (p.third_party_membership IS NULL OR p.third_party_membership = FALSE) THEN 'UNICOM'
              WHEN p.product_type = 6 THEN 'PACKAGE'
              ELSE NULL END AS product_type,
              vp.msrp,
              vp.purchase_price
              FROM datalake_ecommerce.dw_srvc_ecommerce_mmc_variant_product vp 
              JOIN datalake_ecommerce.dw_srvc_ecommerce_mmc_product p
              ON vp.product_code = p.code
              WHERE product_type in (1, 6) OR (product_type = 2 AND (p.third_party_membership IS NULL OR p.third_party_membership = FALSE))
              """.format(db = db,
                         mmc_product = table_names["mmc_product"]))

In [0]:
def write_mmc_product_attribute_table(spark, run_date, db, table_names):
    # Update vin-user-product pair table filtered by candidate products. Set rating=4 
    spark.sql("""
              INSERT OVERWRITE TABLE {db}.{mmc_product_attributes} PARTITION (partition_date = '{run_date}')
              WITH mmc_product_card_uv_imps AS (
                  SELECT card.product_id,
                  p.product_type,
                  count(distinct user_id) as uv_impressions
                  FROM (
                      SELECT user_id,
                      coalesce(product_id, id) as product_id -- Either product_id or id is not null but not both
                      FROM datalake_mystar.dw_cust_dwd_mystar_app_card_show_i_d
                      WHERE dt = '{run_date}'
                      AND user_id != ''
                      AND page_name IN ('store_home', 'mmc_store')
                      AND element = 'product'
                      AND (type is NULL OR type = 'MMC')
                  ) card
                  INNER JOIN {db}.{mmc_product} p
                  ON card.product_id = p.product_id
                  group by 1, 2
              ),
              mmc_product_detail_uv_imps AS (
                  SELECT 
                        pdp.product_id,
                        p.product_type,
                        count(distinct pdp.user_id) as uv_impressions
                  FROM (
                          select user_id,
                          product_id
                          from bz_cn_dl.datalake_mystar.dw_cust_dwd_mystar_app_page_show_i_d
                          where page_name = 'mmc_store_product_detail'
                          and user_id is not null and user_id != ''
                          and dt = '{run_date}'
                        ) pdp
                  INNER JOIN {db}.{mmc_product} p
                  ON pdp.product_id = p.product_id
                  group by 1, 2
              ),
              mmc_product_orders AS (
                    SELECT p.product_id,
                    p.product_type,
                    count(1) as tot_order_cnt,  
                    count_if(o.ordersource = 'MY_STAR') as app_order_cnt,
                    avg(o.msrp) as avg_msrp,
                    avg(o.baseprice) as avg_baseprice,
                    avg(o.totalprice) as avg_trans_price
                    FROM (
                        SELECT customerpk as user_id,
                        CASE WHEN INSTR(productpk, ':') > 0 THEN SUBSTRING(productpk, 1, INSTR(productpk, ':')-1)
                        ELSE productpk END AS variant_id,
                        baseprice,
                        msrp,
                        totalprice,
                        ordersource
                        FROM datalake_ecommerce.dw_cust_ecommerce_dcp_mmc_order 
                        WHERE orderid LIKE '%-%'
                        --For package items (order id ends with a '-0'), the items it contains also have individual order ids. Here when counting orders, consider both sub-items in a package and the package order itself. 
                        AND orderstatus in ('PAID','COMPLETED','WAITING_FOR_CONFIRM','PROCESSING')
                        AND to_date(orderdate) = '{run_date}'
                    ) o
                    INNER JOIN {db}.{mmc_product} p
                    ON o.variant_id = p.variant_id
                    GROUP BY 1, 2
              )
              SELECT p.product_id, 
              p.product_type, 
              coalesce(c.uv_impressions,0) as card_uv_imps, 
              coalesce(d.uv_impressions,0) as pdp_uv_imps, 
              coalesce(o.tot_order_cnt, 0) as tot_order_cnt,
              coalesce(o.app_order_cnt, 0) as app_order_cnt,
              coalesce(o.avg_msrp, 0) as avg_order_msrp,
              coalesce(o.avg_baseprice, 0) as avg_order_baseprice,
              coalesce(o.avg_trans_price, 0) as avg_trans_price
              FROM (
                SELECT distinct product_id, product_type
                FROM {db}.{mmc_product}
              ) p
              LEFT JOIN mmc_product_card_uv_imps c
              ON c.product_id = p.product_id AND c.product_type = p.product_type
              LEFT JOIN mmc_product_detail_uv_imps d
              ON d.product_id = p.product_id AND d.product_type = p.product_type
              LEFT JOIN mmc_product_orders o
              ON o.product_id = p.product_id AND o.product_type = p.product_type
              """.format(db = db,
                        mmc_product = table_names["mmc_product"],
                        mmc_product_attributes = table_names["mmc_product_attributes"],
                        run_date = run_date))


In [0]:
def write_mmc_user_attribute_table(spark, run_date, db, table_names):
    spark.sql("""
              INSERT OVERWRITE TABLE {db}.{mmc_user_attributes} PARTITION (partition_date = '{run_date}')
              with store_visits as (
                select
                    user_id,
                    count(distinct session_id) as store_visits,
                    mode(coalesce(platform, 'unknown')) as platform
                from bz_cn_dl.datalake_mystar.dw_cust_dwd_mystar_app_page_show_i_d
                where
                    page_name IN ('store_home', 'mmc_store')
                    and user_id is not null
                    and user_id != ''
                    and dt = '{run_date}'
                group by 1
                ),
                pdp_views as (
                select
                    user_id,
                    count(*) as pdp_views
                from bz_cn_dl.datalake_mystar.dw_cust_dwd_mystar_app_page_show_i_d
                where
                    page_name = 'mmc_store_product_detail'
                    and user_id is not null
                    and user_id != ''
                    and dt = '{run_date}'
                group by 1
                ),
                orders_placed as (
                SELECT
                    customerpk as user_id,
                    count(*) as tot_items_ordered,
                    count_if(ordersource = 'MY_STAR') as app_items_ordered
                FROM datalake_ecommerce.dw_cust_ecommerce_dcp_mmc_order
                WHERE
                    orderid LIKE '%-%'
                    AND orderid NOT LIKE '%-0' 
                    --There exists multi-item orders. Exclude master orders (order id w/o a '-') and include sub orders or single-item orders only (order id w/ a '-'), For package items (order id ends with a '-0'), the items it contains also have individual order ids. Here when counting orders, consider sub-items in a package but exclude the package order itself. 
                    AND orderstatus in ('PAID', 'COMPLETED', 'WAITING_FOR_CONFIRM', 'PROCESSING')
                    AND to_date(orderdate) = '{run_date}'
                GROUP BY 1
                ),
                coupons as (
                    select vm.ciamid as user_id, 
                    count(distinct coupon_code) filter (where c.status = 'CREATED' and c.start_date = '{run_date}') as coupons_owned,
                    count(distinct coupon_code) filter (where c.status = 'REDEEMED' and c.redeemed_date = '{run_date}') as coupons_redeemed
                    from {db}.{mmc_coupons} c
                    inner join {db}.{vin_meid_mapping} vm 
                    on c.vin = vm.vin
                    where c.partition_date <= '{run_date}'
                    group by 1
                )
                select
                    sv.*,
                    coalesce(pv.pdp_views, 0) as pdp_views,
                    coalesce(op.tot_items_ordered, 0) as tot_items_ordered,
                    coalesce(op.app_items_ordered, 0) as app_items_ordered,
                    coalesce(c.coupons_owned, 0) as coupons_owned,
                    coalesce(c.coupons_redeemed, 0) as coupons_redeemed
                from
                store_visits sv
                left join pdp_views pv
                on sv.user_id = pv.user_id
                left join orders_placed op
                on sv.user_id = op.user_id
                left join coupons c
                on sv.user_id = c.user_id
                """.format(db = db,
                            mmc_user_attributes = table_names["mmc_user_attributes"],
                            mmc_coupons = table_names["mmc_coupons"],
                            vin_meid_mapping = table_names["vin_meid_mapping"],
                            run_date = run_date))

In [0]:
def write_user_product_interaction_table(spark, run_date, db, table_names): 
    spark.sql("""
              INSERT OVERWRITE TABLE {db}.{mmc_user_product_interaction} PARTITION (partition_date = '{run_date}')
              with user_set as (
                select distinct user_id
                from bz_cn_dl.datalake_mystar.dw_cust_dwd_mystar_app_page_show_i_d
                where page_name IN ('store_home', 'mmc_store')
                  and user_id is not null
                  and user_id != ''
                  and dt = '{run_date}'
              ),
              product_card_imp AS (
                SELECT user_id,
                coalesce(product_id, id) as product_id, -- Either product_id or id is not null but not both
                'card_impression' as interaction
                FROM datalake_mystar.dw_cust_dwd_mystar_app_card_show_i_d
                WHERE dt = '{run_date}'
                AND user_id != ''
                AND page_name IN ('store_home', 'mmc_store')
                AND element = 'product'
                AND (type is NULL OR type = 'MMC') --type = MMC for store_home and type=null for mmc_store 
                and product_id in (select distinct product_id from {db}.{mmc_product})
              ),
              view_product_details as (
                select user_id, product_id, 'view_detail' as interaction
                from bz_cn_dl.datalake_mystar.dw_cust_dwd_mystar_app_page_show_i_d
                where page_name = 'mmc_store_product_detail'
                and dt = '{run_date}'
                and user_id in (select user_id from user_set)
                and product_id in (select distinct product_id from {db}.{mmc_product})
              ), 
              mmc_store_order_confirm_exploded_variants_show AS (
                SELECT user_id,
                EXPLODE(SPLIT(CASE 
                              WHEN product_id LIKE '[%]' THEN 
                              SUBSTRING(product_id, INSTR(product_id , '[') + 1, INSTR(product_id , ']') - INSTR(product_id , '[') - 1)
                              ELSE product_id
                              END, 
                              ',')) AS variant_id
                FROM bz_cn_dl.datalake_mystar.dw_cust_dwd_mystar_app_page_show_i_d
                WHERE page_name = 'mmc_store_order_confirm'
                and dt = '{run_date}'
                and user_id in (select user_id from user_set)    
                ),
              mmc_store_order_confirm_exploded_variants_and_products_show AS (
                SELECT user_id,
                TRIM(variant_id) AS variant_id,
                TRIM(SPLIT(variant_id, '-')[0]) AS product_id
                FROM mmc_store_order_confirm_exploded_variants_show 
              ),
              mmc_store_payment_exploded_variants_show AS (
                SELECT user_id,
                EXPLODE(SPLIT(CASE 
                              WHEN product_id LIKE '[%]' THEN 
                              SUBSTRING(product_id, INSTR(product_id , '[') + 1, INSTR(product_id , ']') - INSTR(product_id , '[') - 1)
                              ELSE product_id
                              END, 
                              ',')) AS variant_id
                FROM bz_cn_dl.datalake_mystar.dw_cust_dwd_mystar_app_page_show_i_d
                WHERE (page_name = 'mmc_store_payment') 
                OR (page_name = 'app_payment_method' AND source IN ('store', 'mall'))
                and dt = '{run_date}'
                and user_id in (select user_id from user_set)
              ),
              mmc_store_payment_exploded_variants_and_products_show AS (
                SELECT user_id,
                TRIM(variant_id) AS variant_id,
                TRIM(SPLIT(variant_id, '-')[0]) AS product_id
                FROM mmc_store_payment_exploded_variants_show 
              ),
              init_orders as (
                select oc.user_id, oc.product_id, 'init_order' as interaction
                from mmc_store_order_confirm_exploded_variants_and_products_show oc
                inner join {db}.{mmc_product} p
                on oc.variant_id = p.variant_id
                union all
                select pay.user_id, pay.product_id, 'init_order' as interaction
                from mmc_store_payment_exploded_variants_and_products_show pay
                inner join {db}.{mmc_product} p
                on pay.variant_id = p.variant_id    
              ),
              mmc_store_payment_exploded_variants_success AS (
                SELECT user_id,
                EXPLODE(SPLIT(CASE 
                              WHEN product_id LIKE '[%]' THEN 
                              SUBSTRING(product_id, INSTR(product_id , '[') + 1, INSTR(product_id , ']') - INSTR(product_id , '[') - 1)
                              ELSE product_id
                              END, 
                              ',')) AS variant_id
                FROM bz_cn_dl.datalake_mystar.dw_cust_dwd_mystar_app_page_show_i_d
                WHERE (page_name = 'mmc_store_payment_success') 
                OR (page_name = 'app_payment_result' AND source IN ('store', 'mall') AND status = 1)
                and dt = '{run_date}'
                and user_id in (select user_id from user_set)
              ),
              mmc_store_payment_exploded_variants_and_products_success AS (
                SELECT user_id,
                TRIM(variant_id) AS variant_id,
                TRIM(SPLIT(variant_id, '-')[0]) AS product_id
                FROM mmc_store_payment_exploded_variants_success 
              ),
              mmc_product_orders AS (
                SELECT o.user_id,
                p.product_id,
                o.totalprice as total_payment
                FROM (
                    SELECT customerpk as user_id,
                    CASE WHEN INSTR(productpk, ':') > 0 THEN SUBSTRING(productpk, 1, INSTR(productpk, ':')-1)
                    ELSE productpk END AS variant_id,
                    totalprice
                    FROM datalake_ecommerce.dw_cust_ecommerce_dcp_mmc_order 
                    WHERE orderid LIKE '%-%'
                    --For package items (order id ends with a '-0'), the items it contains also have individual order ids. Here when counting orders, consider both sub-items in a package and the package order itself. 
                    AND orderstatus in ('PAID','COMPLETED','WAITING_FOR_CONFIRM','PROCESSING')
                    AND to_date(orderdate) = '{run_date}'
                    AND ordersource = 'MY_STAR'
                ) o
                INNER JOIN {db}.{mmc_product} p
                ON o.variant_id = p.variant_id
                WHERE user_id in (select user_id from user_set)
              ),
              place_orders AS (
                select ps.user_id, ps.product_id, 'place_order' as interaction, 0 as total_payment
                from mmc_store_payment_exploded_variants_and_products_success ps
                INNER JOIN {db}.{mmc_product} p
                ON ps.variant_id = p.variant_id
                UNION ALL
                select user_id, product_id, 'place_order' as interaction, total_payment
                from mmc_product_orders
              )
              select user_id, 
              product_id, 
              count_if(interaction='card_impression') as card_impressions,
              count_if(interaction='view_detail') as details_viewed,
              count_if(interaction='init_order') as orders_initiated,
              count_if(interaction='place_order') as orders_placed,
              sum(total_payment) as total_payment
              from (
                select *, 0 as total_payment
                from product_card_imp
                union all
                select *, 0 as total_payment
                from view_product_details
                union all
                select *, 0 as total_payment
                from init_orders
                union all
                select *
                from place_orders
              )
              group by 1,2
              """.format(db = db,
                         mmc_user_product_interaction = table_names["mmc_user_product_interaction"],
                         mmc_product = table_names["mmc_product"],
                         run_date = run_date))

In [0]:
import json
import pandas as pd

# Get pipeline parameters
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")

print("Input parameter: start_date {}".format(start_date))
print("Input parameter: end_date {}".format(end_date))

config_path = dbutils.widgets.get("table_config_path")
with open(config_path, "r") as file:
    config = json.load(file)
db, table_names = config['DATABASE'], config['TABLE_NAMES']

In [0]:
print("Writing mmc product list table...")
write_mmc_product_list_table(spark, db, table_names)

date_range = [x.strftime('%Y-%m-%d') for x in pd.date_range(start_date, end_date)]
for run_date in date_range:
    print(f"Running for date: {run_date}")

    print("Writing vin-meid mapping table...")
    write_vin_meid_mapping_table(spark, run_date, db, table_names)

    print("Writing coupon table...")
    write_coupon_table(spark, run_date, db, table_names)

    print("Writing mmc product attribute table...")
    write_mmc_product_attribute_table(spark, run_date, db, table_names)

    print("Writing user attribute table...")
    write_mmc_user_attribute_table(spark, run_date, db, table_names)

    print("Writing user product interaction table...")
    write_user_product_interaction_table(spark, run_date, db, table_names)